# Model Comparison — Chronic Absenteeism

Runs the same model zoo against **three different targets** built from `AttRate`:

| target_col | task | values |
|---|---|---|
| `chronic_absent` | binary classification | 0 / 1 |
| `absence_tier` | 3-class classification | 0 = satisfactory (≥0.95), 1 = at-risk (0.90-0.95), 2 = chronic (<0.90) |
| `att_rate_target` | regression | continuous AttRate |

Linear/distance models use an imputed + standardized matrix; tree models use the NaN-preserving matrix.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from feature_engineering_final import build_all_features, make_model_matrices

train_csv = ROOT / 'data' / 'train_features_final.csv'
test_csv = ROOT / 'data' / 'test_features_final.csv'
if not train_csv.exists():
    build_all_features()
train = pd.read_csv(train_csv)
test = pd.read_csv(test_csv)
print(f'train={train.shape}  test={test.shape}')

train=(88386, 89)  test=(35917, 89)


## Shared evaluators

In [2]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    mean_absolute_error, mean_squared_error, r2_score)

def eval_binary(name, y_true, y_pred, y_prob):
    return {'model': name,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc_roc': roc_auc_score(y_true, y_prob),
            'pr_auc': average_precision_score(y_true, y_prob)}

def eval_multiclass(name, y_true, y_pred, y_prob):
    return {'model': name,
            'accuracy': accuracy_score(y_true, y_pred),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'auc_ovr': roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')}

def eval_regression(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    yt_bin = (y_true < 0.90).astype(int)
    yp_bin = (y_pred < 0.90).astype(int)
    return {'model': name,
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': rmse,
            'r2': r2_score(y_true, y_pred),
            'recall@0.90': recall_score(yt_bin, yp_bin, zero_division=0),
            'precision@0.90': precision_score(yt_bin, yp_bin, zero_division=0)}

## Runners — fit the full zoo for one target

In [3]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier, XGBRegressor

def run_classification(target_col, is_binary):
    Xtr_tree, Xte_tree, Xtr_lin, Xte_lin, ytr, yte, fn, _ = make_model_matrices(
        train, test, target_col=target_col, save_preprocessor=False)
    results = []
    evaluator = eval_binary if is_binary else eval_multiclass

    def fit(name, model, Xtr, Xte):
        model.fit(Xtr, ytr)
        yp = model.predict(Xte)
        yq = model.predict_proba(Xte)
        yq_use = yq[:, 1] if is_binary else yq
        results.append(evaluator(name, yte, yp, yq_use))

    fit('Dummy (stratified)', DummyClassifier(strategy='stratified', random_state=42), Xtr_lin, Xte_lin)
    fit('Logistic (L2)', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), Xtr_lin, Xte_lin)
    fit('Logistic (ElasticNet)', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, class_weight='balanced', max_iter=2000, random_state=42), Xtr_lin, Xte_lin)
    fit('Linear SVC (calibrated)', CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42), cv=3), Xtr_lin, Xte_lin)
    fit('KNN (k=25)', KNeighborsClassifier(n_neighbors=25, n_jobs=-1), Xtr_lin, Xte_lin)

    fit('Decision Tree', Pipeline([('imp', SimpleImputer(strategy='median')),
        ('m', DecisionTreeClassifier(class_weight='balanced', max_depth=8, random_state=42))]), Xtr_tree, Xte_tree)
    fit('Random Forest', Pipeline([('imp', SimpleImputer(strategy='median')),
        ('m', RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=42, n_jobs=-1))]), Xtr_tree, Xte_tree)
    fit('HistGradientBoosting', HistGradientBoostingClassifier(max_iter=300, class_weight='balanced', random_state=42), Xtr_tree, Xte_tree)

    if is_binary:
        ratio = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
        xgb_model = XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss',
            n_estimators=300, random_state=42, n_jobs=-1)
    else:
        xgb_model = XGBClassifier(objective='multi:softprob',
            num_class=int(pd.Series(ytr).nunique()), eval_metric='mlogloss',
            n_estimators=300, random_state=42, n_jobs=-1)
    fit('XGBoost', xgb_model, Xtr_tree, Xte_tree)

    try:
        from lightgbm import LGBMClassifier
        fit('LightGBM', LGBMClassifier(class_weight='balanced', n_estimators=500,
            random_state=42, n_jobs=-1, verbose=-1), Xtr_tree, Xte_tree)
    except ImportError:
        pass

    return pd.DataFrame(results)

def run_regression():
    Xtr_tree, Xte_tree, Xtr_lin, Xte_lin, ytr, yte, fn, _ = make_model_matrices(
        train, test, target_col='att_rate_target', save_preprocessor=False)
    results = []

    def fit(name, model, Xtr, Xte):
        model.fit(Xtr, ytr)
        results.append(eval_regression(name, yte, model.predict(Xte)))

    fit('Dummy (mean)', DummyRegressor(strategy='mean'), Xtr_lin, Xte_lin)
    fit('Ridge', Ridge(alpha=1.0, random_state=42), Xtr_lin, Xte_lin)
    fit('ElasticNet', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=42), Xtr_lin, Xte_lin)
    fit('KNN (k=25)', KNeighborsRegressor(n_neighbors=25, n_jobs=-1), Xtr_lin, Xte_lin)
    fit('Decision Tree', Pipeline([('imp', SimpleImputer(strategy='median')),
        ('m', DecisionTreeRegressor(max_depth=8, random_state=42))]), Xtr_tree, Xte_tree)
    fit('Random Forest', Pipeline([('imp', SimpleImputer(strategy='median')),
        ('m', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))]), Xtr_tree, Xte_tree)
    fit('HistGradientBoosting', HistGradientBoostingRegressor(max_iter=300, random_state=42), Xtr_tree, Xte_tree)
    fit('XGBoost', XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1), Xtr_tree, Xte_tree)
    try:
        from lightgbm import LGBMRegressor
        fit('LightGBM', LGBMRegressor(n_estimators=500, random_state=42, n_jobs=-1, verbose=-1), Xtr_tree, Xte_tree)
    except ImportError:
        pass
    return pd.DataFrame(results)

## Target 1 — Binary (`chronic_absent`)

In [ ]:
res_binary = run_classification('chronic_absent', is_binary=True)\
    .sort_values('auc_roc', ascending=False).reset_index(drop=True)
display(res_binary.style.format({c: '{:.4f}' for c in res_binary.columns if c != 'model'})\
    .background_gradient(cmap='Blues', subset=['auc_roc', 'f1', 'recall']))

## Target 2 — 3-class (`absence_tier`: satisfactory / at-risk / chronic)

In [ ]:
res_multi = run_classification('absence_tier', is_binary=False)\
    .sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(res_multi.style.format({c: '{:.4f}' for c in res_multi.columns if c != 'model'})\
    .background_gradient(cmap='Greens', subset=['f1_macro', 'auc_ovr', 'accuracy']))

## Target 3 — Regression (`att_rate_target`)

In [ ]:
res_reg = run_regression().sort_values('rmse').reset_index(drop=True)
display(res_reg.style.format({c: '{:.4f}' for c in res_reg.columns if c != 'model'})\
    .background_gradient(cmap='Oranges', subset=['rmse', 'mae', 'r2']))

## Summary — leaders across tasks

In [ ]:
summary = pd.DataFrame([
    {'task': 'binary (chronic_absent)', 'best_model': res_binary.iloc[0]['model'],
     'key_metric': 'AUC-ROC', 'value': res_binary.iloc[0]['auc_roc']},
    {'task': '3-class (absence_tier)', 'best_model': res_multi.iloc[0]['model'],
     'key_metric': 'macro-F1', 'value': res_multi.iloc[0]['f1_macro']},
    {'task': 'regression (AttRate)', 'best_model': res_reg.iloc[0]['model'],
     'key_metric': 'RMSE', 'value': res_reg.iloc[0]['rmse']},
])
summary